# Gemma 4 Exploration — `gemma4:latest`\n\nDeep dive on the local `gemma4:latest` model (8.0B, Q4_K_M), which reports capabilities: `completion`, `vision`, `audio`, `tools`, `thinking`.\n\nCovers chat, the `think` reasoning mode, tool calling, and vision (reading the receipts in `notebooks/receipts/` for comparison against the OCR-specialist model in `ocr_exploration.ipynb`).\n\nMake sure the Ollama server is running (`ollama serve`) before executing cells.

In [1]:
import base64
import json
from pathlib import Path

import ollama
import requests

BASE_URL = "http://localhost:11434"
MODEL = "gemma4:latest"

client = ollama.Client(host=BASE_URL)

def pretty(obj):
    print(json.dumps(obj, indent=2, default=str))

## Model details — `POST /api/show`

In [2]:
resp = requests.post(f"{BASE_URL}/api/show", json={"model": MODEL})
info = resp.json()
print("capabilities:", info.get("capabilities"))
print("family:", info["details"]["family"])
print("parameter_size:", info["details"]["parameter_size"])
print("quantization:", info["details"]["quantization_level"])

capabilities: ['completion', 'vision', 'audio', 'tools', 'thinking']
family: gemma4
parameter_size: 8.0B
quantization: Q4_K_M


## Chat — `POST /api/chat`

In [3]:
messages = [
    {"role": "system", "content": "You are a terse assistant. Answer in one or two sentences."},
    {"role": "user", "content": "What sets Gemma models apart from other open-weight LLM families?"},
]

response = client.chat(model=MODEL, messages=messages)
print(response["message"]["content"])

Gemma models are notable for being built upon Google DeepMind's advanced research, leveraging technology derived from Gemini. This blend offers state-of-the-art performance combined with an emphasis on responsible AI and efficiency for broad deployment across various platforms.


### Thinking mode

`gemma4` reports the `thinking` capability — passing `think: true` makes the server return a separate `message.thinking` field with the model's reasoning trace, ahead of `message.content`.

In [4]:
response = client.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "A farmer has 17 sheep, all but 9 die. How many are left?"}],
    think=True,
)
print("--- thinking ---")
print(response["message"].get("thinking"))
print("\n--- answer ---")
print(response["message"]["content"])

--- thinking ---
Thinking Process:

1.  **Analyze the request:** The user is presenting a classic riddle/trick question about surviving animals.
2.  **Identify the key information:**
    *   Starting number of sheep: 17.
    *   Phrase describing loss: "all but 9 die."
3.  **Interpret the phrase "all but X":** The phrase "all but N" means that (Total - N) died, OR it is a tricky way of stating how many *remain*.
4.  **Determine who is left:** If only 9 are *not* dead when everything else dies, then 9 remain alive.
5.  **Formulate the answer:** The number remaining is 9. (The fact that 17 started is extra information meant to distract from the literal meaning of "all but 9 die.")

--- answer ---
There are **9** left.


## Tool calling

In [5]:
def get_exchange_rate(base: str, quote: str) -> str:
    return f"1 {base} = 0.92 {quote} (mock rate)"

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Get the exchange rate between two currencies",
            "parameters": {
                "type": "object",
                "properties": {
                    "base": {"type": "string", "description": "Base currency code, e.g. USD"},
                    "quote": {"type": "string", "description": "Quote currency code, e.g. EUR"},
                },
                "required": ["base", "quote"],
            },
        },
    }
]

response = client.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "What's the USD to EUR exchange rate?"}],
    tools=tools,
)

tool_calls = response["message"].get("tool_calls")
if tool_calls:
    for call in tool_calls:
        print("model wants to call:", call["function"]["name"], call["function"]["arguments"])
        result = get_exchange_rate(**call["function"]["arguments"])
        print("tool result:", result)
else:
    print(response["message"]["content"])

model wants to call: get_exchange_rate {'base': 'USD', 'quote': 'EUR'}
tool result: 1 USD = 0.92 EUR (mock rate)


## Vision — reading the receipts

`gemma4` is a general-purpose vision-language model, unlike the OCR specialist in `ocr_exploration.ipynb`. Same receipts, same structured-extraction prompt — useful for comparing output quality/formatting between the two.

In [6]:
RECEIPTS_DIR = Path("receipts")
receipt_paths = sorted(RECEIPTS_DIR.glob("*.[jp][pn]g"))

EXTRACTION_PROMPT = (
    "Extract this receipt as JSON with keys: "
    '"merchant", "date", "items" (list of {"name", "price"}), "total". '
    "Respond with only the JSON object, no commentary."
)

for path in receipt_paths:
    image_b64 = base64.b64encode(path.read_bytes()).decode()
    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "user", "content": EXTRACTION_PROMPT, "images": [image_b64]}
        ],
    )
    print(f"=== {path.name} ===")
    print(response["message"]["content"])
    print()

=== 1*N9w_Ck211Lo22lYbTd14aQ.jpg ===
```json
{
  "merchant": "CHAUNCEY'S STEAK RESTAURANT",
  "date": "March 12, 2025",
  "items": [
    {
      "name": "Ribeye Steak",
      "price": "$84.00"
    },
    {
      "name": "House Salad",
      "price": "$20.00"
    },
    {
      "name": "Soft Drinks",
      "price": "$10.00"
    }
  ],
  "total": "$132.45"
}
```



=== FE1DgHAWYAAwMgV.jpg ===
```json
{
  "merchant": null,
  "date": "03/11/2021",
  "items": [
    {
      "name": "GOLD PLATED CARRIER",
      "price": 500.00
    },
    {
      "name": "GUCCI GRAPES",
      "price": 595.00
    },
    {
      "name": "POSH PARADISE",
      "price": 1382.00
    },
    {
      "name": "KYLIE SALMON",
      "price": 688.00
    }
  ],
  "total": 1480.75
}
```



=== Fake-Hotel-Receipt-Template.jpg ===
```json
{
  "merchant": null,
  "date": "2014-09-22",
  "items": [
    {
      "name": "Accommodation (09-18-14)",
      "price": 0.00
    },
    {
      "name": "Lodging Tax (09-18-14)",
      "price": 0.00
    },
    {
      "name": "City Tax (09-18-14)",
      "price": 0.00
    },
    {
      "name": "Accommodation (09-19-14)",
      "price": 0.00
    },
    {
      "name": "Lodging Tax (09-19-14)",
      "price": 0.00
    },
    {
      "name": "City Tax (09-19-14)",
      "price": 0.00
    }
  ],
  "total": 903.28
}
```



=== fast-food-receipt-for-ice-cream-frozen-yogurt-custard-store1.png ===
```json
{
  "merchant": "ICE Cream - Frozen Yogurt",
  "date": "05/24/21",
  "items": [
    {
      "name": "Regular Concrete",
      "price": "$5.50"
    },
    {
      "name": "Special Oreo Concrete",
      "price": "$13.00"
    }
  ],
  "total": "$22.62"
}
```



=== fast-food-restaurant-template-with-itemized-food-and-tax.png ===
```json
{
  "merchant": "Fish & Chips Fast Foods",
  "date": "12-01-2020",
  "items": [
    {
      "name": "Fish Burger",
      "price": 25.98
    },
    {
      "name": "Fish & Chips",
      "price": 8.99
    },
    {
      "name": "Soft Drink",
      "price": 3.98
    }
  ],
  "total": 41.29
}
```



## Notes

- `gemma4` reports `audio` as a capability too, but that's not exercised here — the Ollama REST API doesn't currently expose an audio input field for it.
- General-purpose vision models like this one tend to *reason* about a receipt (inferring totals, normalizing dates) rather than transcribing verbatim — compare against the raw-transcription behavior of `deepseek-ocr` in `ocr_exploration.ipynb`.
- Full endpoint reference: https://github.com/ollama/ollama/blob/main/docs/api.md